In [1]:
!pip install diagrams graphviz

  Using cached diagrams-0.24.4-py3-none-any.whl.metadata (7.3 kB)
  Using cached graphviz-0.20.3-py3-none-any.whl.metadata (12 kB)
  Using cached cfgv-3.4.0-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached identify-2.6.15-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached nodeenv-1.9.1-py2.py3-none-any.whl.metadata (21 kB)
Using cached diagrams-0.24.4-py3-none-any.whl (27.8 MB)
Using cached graphviz-0.20.3-py3-none-any.whl (47 kB)
Using cached cfgv-3.4.0-py2.py3-none-any.whl (7.2 kB)
Using cached identify-2.6.15-py2.py3-none-any.whl (99 kB)
Using cached nodeenv-1.9.1-py2.py3-none-any.whl (22 kB)
  Attempting uninstall: graphviz
    Found existing installation: graphviz 0.21
    Uninstalling graphviz-0.21:
      Successfully uninstalled graphviz-0.21

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
from diagrams import Diagram, Cluster, Edge
from diagrams.aws.database import RDS
from diagrams.custom import Custom
from diagrams.programming.framework import Fastapi, React
from diagrams.programming.language import Python
from diagrams.onprem.client import User as OnPremUser

RESOURCES_DIR = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks"
os.makedirs(RESOURCES_DIR, exist_ok=True)

telegram_icon = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks/image.png"
deepseek_icon = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks/image copy.png"
whisper_icon = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks/image copy 2.png"
catboost_icon = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks/image copy 3.png"
shap_icon = "/Users/stepansepilov/Desktop/Code/hack/src/notebooks/image copy 4.png"

required_icons = [telegram_icon, deepseek_icon, whisper_icon, catboost_icon, shap_icon]
missing_icons = [icon for icon in required_icons if not os.path.exists(icon)]

if missing_icons:
    print("!!! ВНИМАНИЕ: Следующие иконки не найдены. Схема может быть неполной. !!!")
    for icon in missing_icons:
        print(f"- {icon}")
    print("--------------------------------------------------------------------------")


with Diagram("Архитектура 'Эхо': Система диагностики выгорания",
             show=False, filename="architecture_diagram",
             direction="TB",
             graph_attr={"splines": "ortho", "fontsize": "12"}):

    # --- Внешние пользователи и сервисы ---
    user = Custom("Сотрудник", telegram_icon)
    manager = OnPremUser("Менеджер")

    with Cluster("Внешние AI Сервисы"):
        llm_service = Custom("DeepSeek\nAPI", deepseek_icon)
        stt_service = Custom("Whisper\n(STT)", whisper_icon)

    with Cluster("Фронтенд"):
        dashboard_ui = React("3D Дашборд\n(React Three Fiber)")

    with Cluster("База данных"):
        db = RDS("SQLite\nDB")

    # --- Бэкенд ---
    with Cluster("Бэкенд (FastAPI)"):
        api_gateway = Fastapi("API\nGateway")

        with Cluster("Сервисный слой"):
            chat_service = Python("Сервис диалогов\n(ChatLM)")
            nlp_service = Python("Сервис анализа\n(NLP)")
            prediction_service = Python("Сервис предсказаний")

        with Cluster("ML Ядро"):
            catboost_model = Custom("CatBoost\nModel", catboost_icon)
            shap_explainer = Custom("SHAP\nExplainer", shap_icon)

        # Внутренние связи
        api_gateway >> Edge(label="→ к Chat", minlen="1") >> chat_service
        api_gateway >> Edge(label="→ к Predict", minlen="1") >> prediction_service

        chat_service >> Edge(label="→ NLP", minlen="1") >> nlp_service
        nlp_service >> Edge(label="→ LLM API", minlen="2") >> llm_service

        prediction_service >> Edge(label="→ CatBoost", minlen="1") >> catboost_model
        catboost_model >> Edge(label="→ SHAP", minlen="1") >> shap_explainer

    # --- Потоки пользователей ---
    user >> Edge(label="Диалог (текст/\nаудио)", minlen="2") >> api_gateway
    api_gateway >> Edge(label="Аудио\n→ STT", minlen="2") >> stt_service
    stt_service >> Edge(label="Распознанный\nтекст", minlen="2") >> chat_service

    chat_service >> Edge(label="R/W История", minlen="2") >> db
    nlp_service >> Edge(label="W Анализ", minlen="2") >> db
    prediction_service >> Edge(label="R Фичи", minlen="2") >> db

    manager >> Edge(label="Просмотр\nотчетов", minlen="2") >> dashboard_ui
    dashboard_ui >> Edge(label="Запрос данных\n(GET)", minlen="2") >> api_gateway

print(f"Диаграмма архитектуры успешно сохранена в файл 'architecture_diagram.png'")
if missing_icons:
    print("Предупреждение: не все кастомные иконки были найдены, на диаграмме могут быть 'пустые' узлы.")

Диаграмма архитектуры успешно сохранена в файл 'architecture_diagram.png'
